In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
import os

In [ ]:

df = pd.read_parquet(r"..\..\artifacts\validation\valid\credit_approval_valid.parquet")
df.head()


In [ ]:
df.info()

In [ ]:
df.drop(
    ["tpep_pickup_datetime", "tpep_dropoff_datetime", "cbd_congestion_fee"],
    axis=1,
    inplace=True,
)

In [ ]:
y = df["total_amount"]
X = df.drop(columns=["total_amount"])

cat_cols = ["VendorID", "store_and_fwd_flag"]
num_cols = X.columns.drop(["VendorID", "store_and_fwd_flag"]).tolist()


num_imputer = SimpleImputer(strategy="mean")
num_scaler = StandardScaler()
num_tr = Pipeline([("impute", num_imputer), ("scale", num_scaler)])

cat_imputer = SimpleImputer(strategy="most_frequent")
ohe = OneHotEncoder()
cat_tr = Pipeline(
    [
        ("impute", cat_imputer),
        ("encode", ohe),
    ]
)

processor = ColumnTransformer(
    [("num", num_tr, num_cols), ("cat", cat_tr, cat_cols)],
    remainder="passthrough",
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_processed = processor.fit_transform(X_train)

X_test_processed = processor.transform(X_test)

In [ ]:
import boto3
import os
# os.chdir("../../")

def push_file_to_s3():
    client = boto3.client("s3")
    client.upload_file(
        "pyproject.toml",
        Bucket="credit-approval-285515884731-eu-north-1-an",
        Key="pyproject.toml",
    )


push_file_to_s3()

In [ ]:
import pandas as pd
import os
import joblib

In [ ]:
X_train_processed, X_test_processed, y_train, y_test = joblib.load(r"../../artifacts/2026-06-20_22-27-01/tranformation/credit_approval_transformed.pkl")


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# -----------------------
# 1. Load data
# -----------------------
data = pd.read_csv("../../artifacts/my_dataset/pusher/credit_approval.csv")  # or paste your dataframe

data.drop("ZipCode", axis=1, inplace=True)
# target
y = data["Approved"]
X = data.drop("Approved", axis=1)

# -----------------------
# 2. Define categorical columns
# -----------------------
cat_cols = [
    "Gender",
    "Married",
    "BankCustomer",
    "Industry",
    "Ethnicity",
    "PriorDefault",
    "Employed",
    "DriversLicense",
    "Citizen",
]

# numeric columns = everything else
num_cols = [c for c in X.columns if c not in cat_cols]

# -----------------------
# 3. Preprocessing
# -----------------------
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols)
], remainder="passthrough")

# -----------------------
# 4. Model
# -----------------------
model = DecisionTreeClassifier(
    max_depth=5,        # prevent overfitting
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# -----------------------
# 5. Train-test split
# -----------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

# -----------------------
# 6. Train
# -----------------------
pipeline.fit(X_train, y_train)

# -----------------------
# 7. Predict
# -----------------------
y_pred = pipeline.predict(X_test)


print(pipeline.named_steps["model"].feature_importances_)

# -----------------------
# 8. Evaluate
# -----------------------
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

[0.         0.0103017  0.         0.         0.         0.00899765
 0.         0.01625016 0.         0.         0.0081082  0.00156527
 0.         0.         0.         0.         0.         0.
 0.         0.         0.70957625 0.0252237  0.         0.
 0.02588827 0.01240687 0.03960062 0.05588785 0.         0.08619349]
Accuracy: 0.7971014492753623
              precision    recall  f1-score   support

           0       0.78      0.82      0.80        68
           1       0.82      0.77      0.79        70

    accuracy                           0.80       138
   macro avg       0.80      0.80      0.80       138
weighted avg       0.80      0.80      0.80       138



In [8]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_

for name, value in zip(feature_names, importances):
    print(f"{name}: {value}")


cat__Gender_1: 0.0
cat__Married_1: 0.010301701037748124
cat__BankCustomer_1: 0.0
cat__Industry_ConsumerDiscretionary: 0.0
cat__Industry_ConsumerStaples: 0.0
cat__Industry_Education: 0.008997645354178034
cat__Industry_Energy: 0.0
cat__Industry_Financials: 0.016250156936628055
cat__Industry_Healthcare: 0.0
cat__Industry_Industrials: 0.0
cat__Industry_InformationTechnology: 0.008108196166614503
cat__Industry_Materials: 0.0015652708485966742
cat__Industry_Real Estate: 0.0
cat__Industry_Research: 0.0
cat__Industry_Transport: 0.0
cat__Industry_Utilities: 0.0
cat__Ethnicity_Black: 0.0
cat__Ethnicity_Latino: 0.0
cat__Ethnicity_Other: 0.0
cat__Ethnicity_White: 0.0
cat__PriorDefault_1: 0.7095762452451843
cat__Employed_1: 0.02522369718781167
cat__DriversLicense_1: 0.0
cat__Citizen_ByOtherMeans: 0.0
cat__Citizen_Temporary: 0.025888265726751594
remainder__Age: 0.012406865458259713
remainder__Debt: 0.0396006204664648
remainder__YearsEmployed: 0.05588784529628797
remainder__CreditScore: 0.0
remainder